# 06 Journalism Search And Retrieval Evaluation

Supplementary product appendix: structured vocabulary powers grounded table discovery. The search system returns source tables and evidence, not numeric answers.


In [1]:
import csv
import json
from pathlib import Path

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def read_json(relative_path: str):
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"missing": str(path)}


def csv_rows(relative_path: str, limit: int | None = None):
    csv.field_size_limit(2_147_483_647)
    path = ROOT / relative_path
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    return rows if limit is None else rows[:limit]


def csv_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    if not path.exists():
        return None
    return len(csv_rows(relative_path))


def parquet_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    return pq.read_table(path).num_rows if path.exists() else None

## Search Index


In [2]:
current = read_json("outputs/search/current_index.json")
summary_path = current.get("summary_path") or current.get("summary")
summary = read_json(summary_path) if summary_path else {}
{
    "current_index": current,
    "summary": summary,
}

{'current_index': {'index_manifest': 'outputs\\search\\run_391c6ea54050920593d9\\index_manifest.json',
  'lexical_index': 'outputs\\search\\run_391c6ea54050920593d9\\lexical.sqlite',
  'run_id': 'run_391c6ea54050920593d9',
  'search_documents': 'data\\processed\\search_documents.parquet',
  'semantic_index': 'outputs\\search\\run_391c6ea54050920593d9\\semantic.faiss',
  'semantic_metadata': 'outputs\\search\\run_391c6ea54050920593d9\\semantic_metadata.parquet'},
 'summary': {}}

## Retrieval Metrics


In [3]:
metrics = read_json("report/retrieval_metrics.json")
{
    "question_count": metrics.get("question_count"),
    "index_run_id": metrics.get("index", {}).get("run_id"),
    "selected_preset": metrics.get("tuning", {}).get("selected_preset"),
    "systems": metrics.get("systems", {}),
}

{'question_count': 200,
 'index_run_id': 'run_391c6ea54050920593d9',
 'selected_preset': 'geography_time_heavy',
 'systems': {'all_vocabulary_bm25': {'original': {'HitRate@1': 0.55,
    'HitRate@10': 0.95,
    'HitRate@5': 0.95,
    'MRR': 0.9225,
    'Relevance@1': 0.75,
    'Relevance@5': 0.975,
    'p50_latency_ms': 164.47379999999612,
    'p95_latency_ms': 339.0973999999005,
    'question_count': 20},
   'reformulated': {'HitRate@1': 0.25,
    'HitRate@10': 0.6,
    'HitRate@5': 0.55,
    'MRR': 0.6876984126984127,
    'Relevance@1': 0.425,
    'Relevance@5': 0.675,
    'p50_latency_ms': 166.7075999998815,
    'p95_latency_ms': 334.81250000022555,
    'question_count': 20}},
  'fused': {'original': {'HitRate@1': 0.5,
    'HitRate@10': 0.95,
    'HitRate@5': 0.9,
    'MRR': 0.9375,
    'Relevance@1': 0.8,
    'Relevance@5': 0.95,
    'p50_latency_ms': 407.78769999997166,
    'p95_latency_ms': 555.7078000001638,
    'question_count': 20},
   'reformulated': {'HitRate@1': 0.35,
    'H